In [16]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"업로드된 파일: {filename}")

Saving 동별 아동인구수.csv to 동별 아동인구수.csv
업로드된 파일: 동별 아동인구수.csv


In [5]:
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip
import random
from google.colab import files

# 1. 격자 데이터 읽기
grid = gpd.read_file("100m격자.shp")
grid = grid.set_crs(epsg=5179, allow_override=True)

# 2. 행정동 GeoJSON 읽기
dong = gpd.read_file("HangJeongDong_ver20250401.geojson")
dong = dong.to_crs(epsg=5179)

# 3. 중심점 기준으로 격자에 동 이름 붙이기
centroids = grid.copy()
centroids["geometry"] = centroids.centroid

# 4. 공간 조인: 중심점이 포함된 동 정보 가져오기
joined = gpd.sjoin(centroids, dong, how="left", predicate="within")

# 5. 다시 Polygon geometry로 복구
joined["geometry"] = grid["geometry"]  # 격자 원래 모양 복원
joined = joined.to_crs(epsg=4326)      # folium용 위경도 변환

# 6. folium 지도 생성
center = [37.547, 127.08]
m = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")

# 7. 동별 색상 지정
dong_names = joined["adm_nm"].dropna().unique()
color_map = {name: f"#{random.randint(0, 0xFFFFFF):06x}" for name in dong_names}

# 8. 지도에 표시
for name in dong_names:
    subset = joined[joined["adm_nm"] == name]
    folium.GeoJson(
        subset,
        style_function=lambda feature, color=color_map[name]: {
            "fillColor": color,
            "color": "black",
            "weight": 0.3,
            "fillOpacity": 0.6,
        },
        tooltip=GeoJsonTooltip(fields=["adm_nm"], aliases=["행정동"])
    ).add_to(m)

# 9. 저장 및 다운로드
m.save("광진구_격자별_행정동지도.html")
files.download("광진구_격자별_행정동지도.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip
import random
from google.colab import files

# 1. 격자 Shapefile 불러오기
grid = gpd.read_file("100m격자.shp").copy()
grid = grid.set_crs(epsg=5179, allow_override=True)

# ✔ gid 없는 경우 생성
# if "gid" not in grid.columns:
#     grid["gid"] = grid.index.astype(str)

# 2. 행정동 GeoJSON 불러오기
dong = gpd.read_file("HangJeongDong_ver20250401.geojson")
dong = dong.to_crs(epsg=5179)

# 3. 중심점 기반 공간 조인을 위해 투영 좌표계 유지 후 중심점 생성
centroids = grid.copy()
centroids["geometry"] = centroids.geometry.centroid

# 4. spatial join (중심점이 포함된 행정동 매핑)
joined = gpd.sjoin(centroids, dong, how="left", predicate="within")

# 5. geometry 복원, 좌표계 WGS84로 변경
joined["geometry"] = grid.geometry.values
joined["gid"] = grid["gid"].values
joined = joined.to_crs(epsg=4326)

# 6. 지도 객체 생성
m = folium.Map(location=[37.547, 127.08], zoom_start=13, tiles="cartodbpositron")

# 7. 행정동별 색상 매핑
dong_names = joined["adm_nm"].dropna().unique()
color_map = {name: f"#{random.randint(0, 0xFFFFFF):06x}" for name in dong_names}

# 8. folium GeoJson 계층 추가
for name in dong_names:
    subset = joined[joined["adm_nm"] == name]
    color = color_map[name]  # 색상 변수 캡처
    folium.GeoJson(
        subset,
        style_function=lambda feature, color=color: {
            "fillColor": color,
            "color": "black",
            "weight": 0.3,
            "fillOpacity": 0.6,
        },
        tooltip=GeoJsonTooltip(fields=["gid", "adm_nm"], aliases=["격자 ID", "행정동"])
    ).add_to(m)

# 9. 저장 및 다운로드
m.save("광진구_격자별_행정동_gid.html")
files.download("광진구_격자별_행정동_gid.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

격자 + 행정동 매핑된 geoJson 파일 만들기

In [38]:
import geopandas as gpd

# 1. 격자 Shapefile 불러오기
grid = gpd.read_file("100m격자.shp").copy()
grid = grid.set_crs(epsg=5179, allow_override=True)

# # ✔ gid 없으면 생성
# if "gid" not in grid.columns:
#     grid["gid"] = grid.index.astype(str)

# 2. 행정동 GeoJSON 불러오기
dong = gpd.read_file("HangJeongDong_ver20250401.geojson")
dong = dong.to_crs(epsg=5179)  # 동일 좌표계로 변환

# 3. 중심점 기반 spatial join
centroids = grid.copy()
centroids["geometry"] = centroids.geometry.centroid
joined = gpd.sjoin(centroids, dong, how="left", predicate="within")

# 4. geometry 복원, 좌표계 WGS84로 변경
joined["geometry"] = grid.geometry.values  # 원래 geometry 복원
joined["gid"] = grid["gid"].values         # gid 유지
joined = joined.to_crs(epsg=4326)

# 5. 필요한 컬럼만 추출 (gid + 행정동명 + geometry)
result = joined[["gid", "adm_nm", "geometry"]].dropna()

# 6. GeoJSON 저장
result.to_file("grid_with_dong.geojson", driver="GeoJSON")

GID별 점수 csv 만들기

In [42]:
import geopandas as gpd
import pandas as pd

# 1. 격자 + 행정동 매핑된 geojson 파일 읽기 (gid 포함된 파일이어야 함!)
joined = gpd.read_file("grid_with_dong.geojson")  # ⚠️ 이 파일에 gid와 adm_nm 둘 다 포함되어야 함

# 2. 동별 아동 인구수 읽기
df_score = pd.read_csv("동별_아동인구수.csv", encoding="euc-kr")

# 3. 점수 계산 (0~10)
df_score["score"] = (df_score["아동인구수(0~9세)"] - df_score["아동인구수(0~9세)"].min()) / \
                    (df_score["아동인구수(0~9세)"].max() - df_score["아동인구수(0~9세)"].min()) * 10
df_score["score"] = df_score["score"]

# 4. 동 이름 정리 및 병합
joined["adm_nm"] = joined["adm_nm"].str.replace("서울특별시 광진구 ", "", regex=False)
merged = joined.merge(df_score[["동", "score"]], left_on="adm_nm", right_on="동", how="left")

# 5. gid + score + dong(행정동 이름) 추출 및 저장
result = merged[["gid", "score", "adm_nm"]].dropna()
result = result.rename(columns={"adm_nm": "dong"})  # 'adm_nm' 컬럼명을 'dong'으로 변경
result.to_csv("GID별_아동인구_점수.csv", index=False)

In [47]:
import geopandas as gpd
import pandas as pd
import folium
from branca.colormap import linear
from folium.features import GeoJsonTooltip
from google.colab import files

# 1. 격자 파일 불러오기
grid = gpd.read_file("100m격자.shp").copy()
grid = grid.set_crs(epsg=5179, allow_override=True)

# 2. 행정동 GeoJSON 불러오기
dong = gpd.read_file("HangJeongDong_ver20250401.geojson")
dong = dong.to_crs(epsg=5179)

# 3. 중심점 기반 조인
centroids = grid.copy()
centroids["geometry"] = centroids.geometry.centroid
joined = gpd.sjoin(centroids, dong, how="left", predicate="within")

# 4. geometry 복원 및 좌표계 변환
joined["geometry"] = grid.geometry.values
joined["gid"] = grid["gid"].values
joined = joined.to_crs(epsg=4326)
joined["adm_nm"] = joined["adm_nm"].str.replace("서울특별시 광진구 ", "", regex=False)
joined = joined[joined["adm_nm"].notna()]

# 5. 점수 불러오기 및 병합
score_df = pd.read_csv("GID별_아동인구_점수.csv", encoding="utf-8").drop_duplicates("dong")
joined = joined.merge(score_df[["dong", "score"]], left_on="adm_nm", right_on="dong", how="left")

# 6. 컬러맵 정의
colormap = linear.Blues_09.scale(0, 10)
colormap.caption = "아동 인구 점수 (0~10)"

# 7. 지도 생성
m = folium.Map(location=[37.547, 127.08], zoom_start=13, tiles="cartodbpositron")

# 8. GeoJson 추가 (색상 = 점수 기반)
folium.GeoJson(
    joined,
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["score"]) if pd.notna(feature["properties"]["score"]) else "gray",
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7,
    },
    tooltip=GeoJsonTooltip(fields=["gid", "adm_nm", "score"], aliases=["격자 ID", "행정동", "점수"])
).add_to(m)

# 9. 컬러바 추가
colormap.add_to(m)

# 10. 저장
m.save("광진구_격자별_아동인구점수_컬러바.html")
files.download("광진구_격자별_아동인구점수_컬러바.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>